In [1]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("../nutritionrater.pdf")
doc = loader.load()
doc

/var/folders/mr/q772k4td40n64z1nlbj0rzyr0000gn/T/ipykernel_4519/746169146.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-09-27T15:03:25+05:30', 'author': 'Priya Mishra', 'moddate': '2026-09-27T15:03:25+05:30', 'title': 'Microsoft Word - nutritionrater', 'source': '../nutritionrater.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}, page_content='Food: SILK Plus Omega-3 DHA, soymilk \nCategory: Legumes and Legume Products \nKey Nutrients: Vitamin D (D2 + D3): 1.2UG; Potassium, K: 144MG; Zinc, Zn: 0.25MG; Selenium, Se: \n2.3UG; Vitamin D (D2 + D3), International Units: 49.0IU; Total lipid (fat): 2.06G; Carbohydrate, by \ndifference: 3.29G; Energy: 45.0KCAL; Water: 91.1G; Total Sugars: 2.47G; Fiber, total dietary: 0.4G; Iron, \nFe: 0.44MG; Magnesium, Mg: 16.0MG; Sodium, Na: 49.0MG; Vitamin A, IU: 412IU \nBrain Health Context: omega-3 fatty acids DHA EPA brain \n \n \n--- \n \nFood: Peanut butter with omega-3, creamy \nCategory: Legumes and Legume Products \nKey Nutrients: Magnesium, Mg: 191MG; Phosphoru

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(doc)

splits

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-09-27T15:03:25+05:30', 'author': 'Priya Mishra', 'moddate': '2026-09-27T15:03:25+05:30', 'title': 'Microsoft Word - nutritionrater', 'source': '../nutritionrater.pdf', 'total_pages': 74, 'page': 0, 'page_label': '1'}, page_content='Food: SILK Plus Omega-3 DHA, soymilk \nCategory: Legumes and Legume Products \nKey Nutrients: Vitamin D (D2 + D3): 1.2UG; Potassium, K: 144MG; Zinc, Zn: 0.25MG; Selenium, Se: \n2.3UG; Vitamin D (D2 + D3), International Units: 49.0IU; Total lipid (fat): 2.06G; Carbohydrate, by \ndifference: 3.29G; Energy: 45.0KCAL; Water: 91.1G; Total Sugars: 2.47G; Fiber, total dietary: 0.4G; Iron, \nFe: 0.44MG; Magnesium, Mg: 16.0MG; Sodium, Na: 49.0MG; Vitamin A, IU: 412IU \nBrain Health Context: omega-3 fatty acids DHA EPA brain \n \n \n--- \n \nFood: Peanut butter with omega-3, creamy \nCategory: Legumes and Legume Products \nKey Nutrients: Magnesium, Mg: 191MG; Phosphoru

In [3]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
db = FAISS.from_documents(splits, OpenAIEmbeddings())
db

/var/folders/mr/q772k4td40n64z1nlbj0rzyr0000gn/T/ipykernel_4519/1738835.py:8: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  db = FAISS.from_documents(splits, OpenAIEmbeddings())


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompts = ChatPromptTemplate.from_template(
    """
    You are helpful AI assistant. Provide the answer based on available context from the document.
    If user query is not matching with the context, just say I don't know.
    
    <context>
    {context}
    </context>
    
    Question: {input}
    """
)


In [5]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")

chain = create_stuff_documents_chain(llm=llm, prompt=prompts)
chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="\n    You are helpful AI assistant. Provide the answer based on available context from the document.\n    If user query is not matching with the context, just say I don't know.\n    \n    <context>\n    {context}\n    </context>\n    \n    Question: {input}\n    "), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-5 Mini', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 1280

In [6]:
from langchain_classic.chains.retrieval import create_retrieval_chain

retrieval_chain = create_retrieval_chain(db.as_retriever(), chain)

retrieval_chain
result = retrieval_chain.invoke({"input": "What are the steps to improve brain health?"})
result['answer']

'Based on the provided context, key steps are:\n\n- Increase antioxidant intake — eat antioxidant-rich foods such as blueberries to help protect brain health.  \n- Include omega‑3 fatty acids (DHA and EPA) — get these from foods or fortified products (the context mentions omega‑3 in relation to breakfast cereals/fortified foods or supplements).  \n- Ensure adequate zinc intake — consume zinc-containing foods (the context links zinc to cognitive function; organ/meat sources are noted).'